In [1]:
import pandas as pd
from pathlib import Path

# Folder containing the Excel files

inputdirectory = '../../22 Fullstream Music/202607. Peter Maffay/data/as received'

input_dir = Path(inputdirectory)

# Sheets are grouped by their column layout: the first group uses the German
# SAP export columns (incl. WHOLESALE_VALUE), the second uses the English
# contract-statement columns (incl. Net Amount). Both groups get combined
# together into the single final aligned output further below.
sheet_groups = {
    "GSA Red Rooster": [
        "digital GSA Red Rooster Katalog",
        "dig.GSA Red Rooster MTV Unpl.",
        "Anouk - Das Musical Q4 2024",
    ],
    "ExGSA and GVL": [
        "dig.Ex-GSA Katalog",
        "dig.Ex-GSA MTV",
        "other income_GVL Katalog",
        "other income_GVL MTV",
    ],
}

sheet_names = [name for names in sheet_groups.values() for name in names]


def year_period_to_month(year_col, period_col):
    """Combine a year column and a 1-12 period column into 'YYYY MM' strings."""
    year = pd.to_numeric(year_col, errors="coerce")
    period = pd.to_numeric(period_col, errors="coerce")
    result = pd.Series(pd.NA, index=year.index, dtype="object")
    mask = year.notna() & period.notna()
    result[mask] = year[mask].astype(int).astype(str) + " " + period[mask].astype(int).astype(str).str.zfill(2)
    return result


def month_to_quarter(month_series):
    """Turn 'YYYY MM' strings into 'YYYY QN' strings."""
    result = pd.Series(pd.NA, index=month_series.index, dtype="object")
    mask = month_series.notna()
    parts = month_series[mask].str.split(" ", expand=True)
    quarters = ((parts[1].astype(int) - 1) // 3 + 1).astype(str)
    result[mask] = parts[0] + " Q" + quarters
    return result


# Loading the raw input files is slow (large workbooks), so this only runs
# once per kernel session. Re-run this cell explicitly (e.g. after a kernel
# restart, or if the input files themselves changed) to reload from disk.
if "sheet_data" not in globals():
    sheet_data = {name: [] for name in sheet_names}

    for file in sorted(input_dir.glob("*.xls*")):
        try:
            available = pd.ExcelFile(file).sheet_names
        except Exception as e:
            print(f"Skipped {file.name}: {e}")
            continue

        present = [name for name in sheet_names if name in available]
        for name in sheet_names:
            if name not in available:
                print(f"'{name}' not found in {file.name}, skipping")

        if not present:
            continue

        # Read all present sheets from this file in one pass (much faster than
        # opening the workbook separately for each sheet)
        sheets = pd.read_excel(file, sheet_name=present)

        for name, df in sheets.items():
            df = df.copy()

            # Drop trailing summary/total rows below the actual data table: real
            # data rows always have the first column populated, while sum/analysis
            # rows appended below the table leave it blank
            first_col = df.columns[0]
            last_valid = df[first_col].last_valid_index()
            if last_valid is None:
                df = df.iloc[0:0]
            elif last_valid < len(df) - 1:
                dropped = len(df) - (last_valid + 1)
                df = df.loc[:last_valid]
                print(f"  Dropped {dropped} trailing summary/blank row(s) from '{name}' in {file.name}")

            # Tabs with WHOLESALE_VALUE don't have ready-made period columns, so
            # derive them from Jahr/Monat (reporting) and EMD_YR/EMD_PERIOD_ID (sales)
            if "WHOLESALE_VALUE" in df.columns:
                df["Reporting Month"] = year_period_to_month(df["Jahr"], df["Monat"])
                df["Reporting Quarter"] = month_to_quarter(df["Reporting Month"])
                df["Sales Month"] = year_period_to_month(df["EMD_YR"], df["EMD_PERIOD_ID"])
                df["Sales Quarter"] = month_to_quarter(df["Sales Month"])

            df.insert(0, "File Name", file.name)
            df.insert(0, "Table Name", name)
            sheet_data[name].append(df)
            print(f"Processed '{name}' from {file.name}: {len(df)} rows")

    print("\nFinished loading input files.")
else:
    print("Input files already loaded in memory; skipping reload (re-run this cell if the input files changed).")




Processed 'digital GSA Red Rooster Katalog' from Red Rooster Q1 2025 Abrechnung_2025.xlsx: 98497 rows
Processed 'dig.GSA Red Rooster MTV Unpl.' from Red Rooster Q1 2025 Abrechnung_2025.xlsx: 7254 rows
  Dropped 6 trailing summary/blank row(s) from 'Anouk - Das Musical Q4 2024' in Red Rooster Q1 2025 Abrechnung_2025.xlsx
Processed 'Anouk - Das Musical Q4 2024' from Red Rooster Q1 2025 Abrechnung_2025.xlsx: 233 rows
Processed 'dig.Ex-GSA Katalog' from Red Rooster Q1 2025 Abrechnung_2025.xlsx: 74097 rows
Processed 'dig.Ex-GSA MTV' from Red Rooster Q1 2025 Abrechnung_2025.xlsx: 6044 rows
Processed 'other income_GVL Katalog' from Red Rooster Q1 2025 Abrechnung_2025.xlsx: 11642 rows
Processed 'other income_GVL MTV' from Red Rooster Q1 2025 Abrechnung_2025.xlsx: 769 rows
'Anouk - Das Musical Q4 2024' not found in Red Rooster Q2 2025 Abrechnung_2025.xlsx, skipping
Processed 'digital GSA Red Rooster Katalog' from Red Rooster Q2 2025 Abrechnung_2025.xlsx: 65790 rows
Processed 'dig.GSA Red Rooste

In [20]:
# Folder with the harmonisation lookup tables (one level above the input files)
lookup_dir = input_dir.parent / "lookup"

# Always reloaded from disk (not cached) so that edits to these files are
# picked up the next time this cell runs
title_lookup_df = pd.read_excel(lookup_dir / "Title.xlsx")
title_lookup = dict(zip(title_lookup_df["Product Title"], title_lookup_df["Product Title (harmonised)"]))

dsp_lookup_df = pd.read_excel(lookup_dir / "DSP.xlsx")
dsp_lookup = dict(zip(dsp_lookup_df["Third Party / DSP"], dsp_lookup_df["Third Party / DSP (Group)"]))


def _normalise_key(value):
    return str(value).strip().casefold()


def harmonise(df, column, new_column, lookup_map, label):
    """Add `new_column` = `column` mapped through `lookup_map` (case/whitespace-
    insensitive), leaving the original `column` untouched. Alerts on any values
    with no match; those rows get an empty `new_column`."""
    normalised_lookup = {_normalise_key(key): value for key, value in lookup_map.items()}

    values = df[column]
    normalised_values = values.map(lambda v: _normalise_key(v) if pd.notna(v) else v)
    is_known = normalised_values.isin(normalised_lookup.keys())
    missing_mask = values.notna() & ~is_known
    if missing_mask.any():
        counts = values[missing_mask].value_counts()
        print(
            f"WARNING: {len(counts)} '{column}' value(s) could not be harmonised "
            f"against {label} ({missing_mask.sum()} rows total):"
        )
        for value, count in counts.items():
            print(f"  {value!r} ({count} rows)")
        print()
    df[new_column] = normalised_values.map(normalised_lookup)
    return df


group_combined_by_label = {}

for group_label, names in sheet_groups.items():
    group_frames = []

    for name in names:
        frames = sheet_data[name]
        if not frames:
            print(f"No data found for sheet '{name}'")
            continue

        # Concatenating with sort=False aligns columns across files (union of
        # all columns), filling any missing ones with NaN
        combined = pd.concat(frames, ignore_index=True, sort=False)

        # Different sheets use different column names for the monetary amount
        if "Net Amount" in combined.columns:
            amount_col = "Net Amount"
        elif "WHOLESALE_VALUE" in combined.columns:
            amount_col = "WHOLESALE_VALUE"
        else:
            amount_col = None

        if amount_col:
            amount_sum = combined[amount_col].sum()
            print(f"'{name}': {len(combined)} rows, {amount_col} sum = {amount_sum}")
        else:
            print(f"'{name}': {len(combined)} rows, no 'Net Amount' or 'WHOLESALE_VALUE' column found")

        group_frames.append(combined)

    if not group_frames:
        print(f"No data found for group '{group_label}'\n")
        continue

    # Aligning columns across the sheets within the group as well
    group_combined = pd.concat(group_frames, ignore_index=True, sort=False)
    group_combined_by_label[group_label] = group_combined
    print(f"'{group_label}' group: {len(group_combined)} total rows\n")

# --- Combine both groups into one final, aligned table ----------------------
# Maps each group's source column names to the shared final column names,
# per the provided column-alignment table
wholesale_rename = {
    "Table Name": "Table Name",
    "File Name": "File Name",
    "INTERPRET": "Artist",
    "TITEL": "Product Title",
    "ISRC": "ISRC",
    "Reporting Quarter": "Reporting Quarter",
    "Reporting Month": "Reporting Month",
    "Sales Quarter": "Sales Quarter",
    "Sales Month": "Sales Month",
    "PARENT_SALES_CAT_DC": "Distribution Channel",
    "SALES_CATEGORY_DC": "Distribution Manner",
    "PROVIDER_NAME": "Third Party / DSP",
    "VENDOR_NAME": "Third Party / DSP (Product)",
    "QUANTITY": "Units",
    "WHOLESALE_VALUE": "Net Amount",
    "COUNTRY_SALE": "Country of Sale",
    "TONTRAEGER_BEZ": "Product Configuration",
}
net_amount_rename = {
    "Table Name": "Table Name",
    "File Name": "File Name",
    "Product Main Artist": "Artist",
    "Product Title": "Product Title",
    "ISRC": "ISRC",
    "Accounting Period": "Reporting Quarter",
    "Reported Period": "Reporting Month",
    "Sales Period": "Sales Quarter",
    "Distribution Channel": "Distribution Channel",
    "Distribution Manner": "Distribution Manner",
    "Third Party / DSP": "Third Party / DSP",
    "Net Invoice Price": "Net Invoice Price",
    "Received Rate %": "Received Rate %",
    "Amount per Unit": "Amount per Unit",
    "SAP Sales Units": "Units",
    "Net Amount": "Net Amount",
    "Country of Sale": "Country of Sale",
    "Product Configuration": "Product Configuration",
}
group_renames = {
    "GSA Red Rooster": wholesale_rename,
    "ExGSA and GVL": net_amount_rename,
}
final_columns = [
    "Table Name",
    "File Name",
    "Artist",
    "Product Title",
    "ISRC",
    "Reporting Quarter",
    "Reporting Month",
    "Sales Quarter",
    "Sales Month",
    "Distribution Channel",
    "Distribution Manner",
    "Third Party / DSP",
    "Third Party / DSP (Product)",
    "Net Invoice Price",
    "Received Rate %",
    "Amount per Unit",
    "Units",
    "Net Amount",
    "Country of Sale",
    "Product Configuration",
]

aligned_frames = []
for group_label, group_combined in group_combined_by_label.items():
    rename_map = group_renames[group_label]

    # Every key in the rename map is expected to exist in the source data;
    # flag any that don't, since that usually means the source sheet's
    # columns changed and the mapping table above is now out of date
    missing_source_columns = [col for col in rename_map if col not in group_combined.columns]
    if missing_source_columns:
        print(f"ERROR: '{group_label}' is missing expected column(s) to rename: {missing_source_columns}")

    aligned = group_combined.rename(columns=rename_map)
    # Keep exactly the final columns, in order; any column not produced by
    # this group's rename map is left empty (NaN)
    aligned = aligned.reindex(columns=final_columns)
    aligned_frames.append(aligned)

final_combined = pd.concat(aligned_frames, ignore_index=True, sort=False)

# Totals computed on the final combined table (i.e. after alignment, so
# "Net Amount" already covers what used to be WHOLESALE_VALUE too)
print(f"Total 'Net Amount' across all sheets: {final_combined['Net Amount'].sum()}")
print(f"Total 'Units' across all sheets: {final_combined['Units'].sum()}")

# Harmonise Product Title and Third Party / DSP against the lookup tables,
# adding new columns alongside the originals (originals are left untouched)
# Column -> new harmonised-column-to-insert-right-after, in insertion order
harmonisations = [
    ("Product Title", "Product Title (harmonised)", title_lookup, "Title.xlsx"),
    ("Third Party / DSP", "Third Party / DSP (Group)", dsp_lookup, "DSP.xlsx"),
]
for column, new_column, lookup_map, label in harmonisations:
    final_combined = harmonise(final_combined, column, new_column, lookup_map, label)

# Place each new harmonised column right next to its original, derived from
# final_columns so the two lists can't drift out of sync
harmonised_column_after = {column: new_column for column, new_column, _, _ in harmonisations}
final_columns_with_harmonised = []
for column in final_columns:
    final_columns_with_harmonised.append(column)
    if column in harmonised_column_after:
        final_columns_with_harmonised.append(harmonised_column_after[column])
final_combined = final_combined.reindex(columns=final_columns_with_harmonised)

output_dir = input_dir.parent / "data modified by JN"
output_dir.mkdir(parents=True, exist_ok=True)
final_output_path = output_dir / "Maffay_2025_combined_harmonised.CSV"
#final_combined.to_csv(final_output_path, index=False)
print(f"'{final_output_path.name}': {len(final_combined)} total rows -> {final_output_path}")

'digital GSA Red Rooster Katalog': 332482 rows, WHOLESALE_VALUE sum = 447907.6175988213
'dig.GSA Red Rooster MTV Unpl.': 26411 rows, WHOLESALE_VALUE sum = 47896.059966864166
'Anouk - Das Musical Q4 2024': 233 rows, WHOLESALE_VALUE sum = 857.51610253518
'GSA Red Rooster' group: 359126 total rows

'dig.Ex-GSA Katalog': 228021 rows, Net Amount sum = 16775.16
'dig.Ex-GSA MTV': 19888 rows, Net Amount sum = 1041.8000000000002
'other income_GVL Katalog': 26412 rows, Net Amount sum = 95008.79000000001
'other income_GVL MTV': 1579 rows, Net Amount sum = 1598.4300000000003
'ExGSA and GVL' group: 275900 total rows

Total 'Net Amount' across all sheets: 611085.373668221
Total 'Units' across all sheets: 131760884.0
'Maffay_2025_combined_harmonised.CSV': 635026 total rows -> ../../22 Fullstream Music/202607. Peter Maffay/data/data modified by JN/Maffay_2025_combined_harmonised.CSV


In [21]:
# --- Pivot table: Net Amount by Sales Month (rows) / Reporting Quarter (columns) ---
pivot = final_combined.pivot_table(
    index="Sales Month",
    columns="Reporting Quarter",
    values="Net Amount",
    aggfunc="sum",
)
print("\nPivot table - 'Net Amount' by Sales Month (rows) / Reporting Quarter (columns):")
print(pivot)

pivot_output_path = output_dir / "pivot.csv"
pivot.to_csv(pivot_output_path)
print(f"'{pivot_output_path.name}' -> {pivot_output_path}")


Pivot table - 'Net Amount' by Sales Month (rows) / Reporting Quarter (columns):
Reporting Quarter     2024 Q4       2025 Q1       2025 Q2       2025 Q3  \
Sales Month                                                               
2018 12                   NaN      0.002424           NaN           NaN   
2019 10                   NaN    613.525798           NaN           NaN   
2020 10                   NaN   2243.771876           NaN           NaN   
2021 01                   NaN      0.066823           NaN           NaN   
2021 09                   NaN     61.555170    -97.468600           NaN   
2021 10                   NaN           NaN      0.000000           NaN   
2021 11                   NaN           NaN     -0.144412           NaN   
2022 02                   NaN    -76.704507           NaN    -30.137253   
2022 05                   NaN           NaN           NaN     27.730631   
2022 08                   NaN           NaN     21.320400    829.229438   
2022 09            